<!-- version-stamp --> **Version** 1.1 · **Released** 2026-09-26 · **Branch** main


<!-- version-stamp --> **Version** 1.1 · **Released** 2026-09-26 · **Branch** main


<!-- version-stamp --> **Version** 1.1 · **Released** 2026-09-26 · **Branch** main


<!-- version-stamp --> **Version** 1.1 · **Released** 2026-09-26 · **Branch** main


<!-- version-stamp --> **Version** 1.1 · **Released** 2026-09-26 · **Branch** main


<!-- version-stamp --> **Version** 1.1 · **Released** 2026-09-26 · **Branch** main


<!-- version-stamp --> **Version** 1.1 · **Released** 2026-09-26 · **Branch** main


<!-- version-stamp --> **Version** 1.1 · **Released** 2026-09-26 · **Branch** main


<!-- version-stamp --> **Version** 1.1 · **Released** 2026-09-26 · **Branch** main


<!-- version-stamp --> **Version** 1.1 · **Released** 2026-09-26 · **Branch** main


<!-- version-stamp --> **Version** 1.1 · **Released** 2026-09-26


# TrustGate experimentation: two side experiments to locate the ceiling

`OBSERVATIONS.md` argues that the main model is limited by the information it is
allowed to see, not by data quantity or by the learning algorithm. This notebook
runs the two experiments that test that claim. Neither is for deployment.

**Experiment 1: author identity.** Add which system wrote the patch as a feature.
If the score jumps, a large part of the missing information is reputation, which
the production design excludes on purpose. The number tells us how large.

**Experiment 2: LLM judge as a ceiling.** Ask a foundation model to read the issue
and the patch and estimate the chance it works, before any test runs. If that
lifts ROC-AUC well past 0.8, the shortfall was information and our approach was
the constraint. If it does not, the label is too noisy to predict from anything
short of running the tests, and a pre-test gate cannot approve unattended at all.

Rules that carry over from the main notebook: same candidate table, same
issue-grouped 60/20/20 split with the same seeds, thresholds chosen on validation
and frozen before test, every headline number with an issue-level bootstrap
interval. This notebook is self-contained: it does not run or import the main notebook and does not read the exported
model. It needs only the candidate table from `./run_pipeline.sh` (add `--static` so the code-health
features are present) and the pipeline modules under `pipeline/`.

In [ ]:
# Setup: same data, same split, same baseline as the main notebook.
import json
import os
import re
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, "pipeline")
from common import CANDIDATE_TABLE_PATH, SPLIT, unified_diff_stats

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupShuffleSplit

plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.grid"] = True
TRUSTGATE_SPLIT = SPLIT
print("Split:", TRUSTGATE_SPLIT, "| table:", CANDIDATE_TABLE_PATH)

df = pd.read_csv(CANDIDATE_TABLE_PATH).reset_index(drop=True)
df = pd.concat([df, pd.DataFrame([unified_diff_stats(p) for p in df["patch"].fillna("")])], axis=1)
# "Author" means the leaderboard submission that produced the patch (agent + model), not a person.
# SWE-bench itself has no author field. candidate_id is "<submission>::<instance_id>" and the
# submission is unique, whereas agent_name can collide (two OpenHands entries with different models).
df["submission"] = df["candidate_id"].str.split("::").str[0]

def grouped_split(data, group_col="instance_id"):
    g1 = GroupShuffleSplit(n_splits=1, train_size=0.60, random_state=42)
    tr, tmp = next(g1.split(data, groups=data[group_col]))
    train, temp = data.iloc[tr].copy(), data.iloc[tmp].copy()
    g2 = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=43)
    va, te = next(g2.split(temp, groups=temp[group_col]))
    return train, temp.iloc[va].copy(), temp.iloc[te].copy()

train, val, test = grouped_split(df)
for name, part in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:>5}: {len(part):>5} candidates, {part['instance_id'].nunique():>4} issues, positive rate {part['label'].mean():.3f}")

BASE13 = ["patch_additions", "patch_deletions", "patch_files", "patch_hunks", "patch_chars",
          "self_consistency", "patch_apply_ok", "syntax_ok", "compile_ok",
          "lint_warning_count", "static_error_count", "undefined_name_count", "import_error_count"]
BASE13 = [f for f in BASE13 if df[f].notna().any()]

def hgb():
    return HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=150, random_state=42)

y_train, y_val, y_test = (part["label"].astype(int).values for part in (train, val, test))

In [ ]:
# Shared evaluation helpers: bootstrap interval, gate thresholds, selection test.
APPROVE_TARGET, REJECT_TARGET, MIN_AUTO_RATE = 0.90, 0.90, 0.05

def grouped_bootstrap_auc(y, p, groups, n_boot=500, seed=42):
    rng = np.random.default_rng(seed)
    frame = pd.DataFrame({"y": y, "p": p, "g": groups})
    by = {g: d for g, d in frame.groupby("g")}
    keys = np.array(list(by))
    vals = []
    for _ in range(n_boot):
        s = pd.concat([by[k] for k in rng.choice(keys, len(keys), replace=True)])
        if s["y"].nunique() == 2:
            vals.append(roc_auc_score(s["y"], s["p"]))
    return float(np.quantile(vals, 0.025)), float(np.quantile(vals, 0.975))

def gate_metrics(y, p, a, r):
    y = np.asarray(y); appr, rej = p >= a, p <= r
    return {"approve_share": appr.mean(), "approve_precision": y[appr].mean() if appr.any() else np.nan,
            "reject_share": rej.mean(), "reject_precision": 1 - y[rej].mean() if rej.any() else np.nan,
            "review_share": 1 - appr.mean() - rej.mean()}

def choose_thresholds(y, p):
    rows = []
    for a in np.arange(0.50, 0.97, 0.02):
        for r in np.arange(0.04, 0.51, 0.02):
            if r < a:
                m = gate_metrics(y, p, a, r); m.update(a=a, r=r); rows.append(m)
    g = pd.DataFrame(rows).fillna({"approve_precision": 0, "reject_precision": 0})
    ok = g[(g.approve_precision >= APPROVE_TARGET) & (g.reject_precision >= REJECT_TARGET)]
    if len(ok):
        best = ok.sort_values("review_share").iloc[0]; met = True
    else:
        u = g[(g.approve_share >= MIN_AUTO_RATE) & (g.reject_share >= MIN_AUTO_RATE)]
        u = u if len(u) else g
        u = u.assign(minp=u[["approve_precision", "reject_precision"]].min(axis=1))
        best = u.sort_values(["minp", "review_share"], ascending=[False, True]).iloc[0]; met = False
    return float(best.a), float(best.r), met

def selection_rate(frame, score_col):
    """Pick the top-scored candidate per issue; share of issues where that pick works."""
    return frame.groupby("instance_id").apply(lambda d: d.loc[d[score_col].idxmax(), "label"]).mean()

def evaluate(name, p_val, p_test):
    lo, hi = grouped_bootstrap_auc(y_test, p_test, test["instance_id"].values)
    a, r, met = choose_thresholds(y_val, p_val)
    g = gate_metrics(y_test, p_test, a, r)
    t = test.assign(_s=p_test)
    return {"experiment": name, "test_auc": roc_auc_score(y_test, p_test), "auc_lo": lo, "auc_hi": hi,
            "test_pr_auc": average_precision_score(y_test, p_test),
            "approve_thr": a, "reject_thr": r, "targets_met": met,
            "approve_share": g["approve_share"], "approve_precision": g["approve_precision"],
            "reject_share": g["reject_share"], "reject_precision": g["reject_precision"],
            "selection_resolve_rate": selection_rate(t, "_s")}

oracle = test.groupby("instance_id")["label"].max().mean()
best_system = test.groupby("submission")["label"].mean().sort_values(ascending=False)
best_system_rate = test.groupby("instance_id").apply(
    lambda d: d.loc[d["submission"] == best_system.index[0], "label"].max() if (d["submission"] == best_system.index[0]).any() else 0).mean()
print(f"Reference points on the test issues: oracle {oracle:.3f}, always-take-{best_system.index[0]} {best_system_rate:.3f}")

## Experiment 1: how much of the gap is reputation?

"Author" here means the leaderboard submission that produced the patch, an agent
framework paired with a model. SWE-bench has no human-author field, and the
submission is the finest identity available; the agent name alone is coarser
because two OpenHands entries with different models share it.

Four models on the same split:

- **Baseline**: the 13 surface-plus-agreement features, the main notebook's set 3.
- **+ one-hot author**: baseline plus an indicator column per system.
- **+ author prior**: baseline plus one number, the system's resolve rate measured on the training issues only. Unseen systems get the global training rate. A cleaner encoding of "reputation" than one-hot, and it cannot leak test information.
- **Author prior alone**: how far reputation gets with no patch information at all.

The last row of the results table is the trivial rule "always take the strongest system", which the production design forbids.

In [ ]:
# Experiment 1: author identity as a feature (side experiment, not for deployment).
def one_hot_agents(part, columns):
    d = pd.get_dummies(part["submission"], prefix="sub").astype(float)
    return d.reindex(columns=columns, fill_value=0.0)

agent_cols = sorted("sub_" + a for a in train["submission"].unique())
prior = train.groupby("submission")["label"].mean()
global_rate = train["label"].mean()
for part in (train, val, test):
    part["agent_prior"] = part["submission"].map(prior).fillna(global_rate)

runs = {}
# Baseline
m = hgb().fit(train[BASE13], y_train)
runs["baseline: 13 surface + agreement"] = (m.predict_proba(val[BASE13])[:, 1], m.predict_proba(test[BASE13])[:, 1])
# + one-hot author
Xtr = pd.concat([train[BASE13].reset_index(drop=True), one_hot_agents(train, agent_cols).reset_index(drop=True)], axis=1)
Xva = pd.concat([val[BASE13].reset_index(drop=True), one_hot_agents(val, agent_cols).reset_index(drop=True)], axis=1)
Xte = pd.concat([test[BASE13].reset_index(drop=True), one_hot_agents(test, agent_cols).reset_index(drop=True)], axis=1)
m = hgb().fit(Xtr, y_train)
runs["+ one-hot author"] = (m.predict_proba(Xva)[:, 1], m.predict_proba(Xte)[:, 1])
# + author prior
F = BASE13 + ["agent_prior"]
m = hgb().fit(train[F], y_train)
runs["+ author prior (train resolve rate)"] = (m.predict_proba(val[F])[:, 1], m.predict_proba(test[F])[:, 1])
# author prior alone
runs["author prior alone"] = (val["agent_prior"].values, test["agent_prior"].values)

exp1 = pd.DataFrame([evaluate(k, *v) for k, v in runs.items()]).set_index("experiment")
exp1.loc["always take the strongest system", ["selection_resolve_rate"]] = best_system_rate
exp1.loc["oracle (any candidate works)", ["selection_resolve_rate"]] = oracle
show = ["test_auc", "auc_lo", "auc_hi", "test_pr_auc", "approve_share", "approve_precision", "reject_share", "reject_precision", "selection_resolve_rate"]
display(exp1[show].round(3))

base_auc = exp1.loc["baseline: 13 surface + agreement", "test_auc"]
prior_auc = exp1.loc["+ author prior (train resolve rate)", "test_auc"]
print(f"\nReputation adds {prior_auc - base_auc:+.3f} ROC-AUC on test "
      f"(baseline interval {exp1.loc['baseline: 13 surface + agreement', 'auc_lo']:.3f} to {exp1.loc['baseline: 13 surface + agreement', 'auc_hi']:.3f}).")

plt.figure(figsize=(8, 3.8))
rows = exp1.dropna(subset=["test_auc"])
plt.errorbar(rows["test_auc"], range(len(rows)), xerr=[rows["test_auc"] - rows["auc_lo"], rows["auc_hi"] - rows["test_auc"]], fmt="o", capsize=5)
plt.yticks(range(len(rows)), rows.index); plt.axvline(0.5, ls="--", lw=0.8)
plt.xlabel("Test ROC-AUC with 95% issue-level bootstrap interval"); plt.title("Experiment 1: author identity")
plt.tight_layout(); plt.show()

In [ ]:
# Experiment 1 reading, written from the numbers above.
b = exp1.loc["baseline: 13 surface + agreement"]; oh = exp1.loc["+ one-hot author"]; pr = exp1.loc["+ author prior (train resolve rate)"]; alone = exp1.loc["author prior alone"]
gain = pr["test_auc"] - b["test_auc"]
outside = gain > (b["auc_hi"] - b["test_auc"])
print("Experiment 1 findings")
print(f"- Baseline test AUC {b['test_auc']:.3f} ({b['auc_lo']:.3f} to {b['auc_hi']:.3f}).")
print(f"- Adding the author's training resolve rate: {pr['test_auc']:.3f}, a change of {gain:+.3f}, "
      f"{'outside' if outside else 'inside'} the baseline's bootstrap half-width.")
print(f"- One-hot author: {oh['test_auc']:.3f}. Author prior alone, no patch information: {alone['test_auc']:.3f}.")
print(f"- Auto-approve precision at validation-chosen thresholds: baseline {b['approve_precision']:.3f} on {b['approve_share']:.1%} of patches; "
      f"with author prior {pr['approve_precision']:.3f} on {pr['approve_share']:.1%}.")
print(f"- Selection test (top pick per issue resolves): baseline {b['selection_resolve_rate']:.3f}, with author prior {pr['selection_resolve_rate']:.3f}, "
      f"always-take-strongest {best_system_rate:.3f}, oracle {oracle:.3f}.")
print("\nInterpretation: the size of the gain is the share of the missing information that is reputation. "
      "It is excluded from the production model on purpose, so this number is a cost of that choice, not a recommendation to include it.")

## Experiment 2: judges that read the code, as the ceiling

A foundation model reads the issue and the patch and returns a probability that
the patch fully resolves the issue. Nothing about tests or outcomes is shown to
it. Design, all in `pipeline/llm_judge.py`:

- Only validation and test candidates are scored. The judge is evaluated on test;
  any combination with TrustGate is fitted on validation. Training rows are never
  needed, which keeps the cost to about 2,000 requests on Verified.
- Requests go through the Message Batches API at half price, with structured JSON
  output and low effort. Scores are cached, so the API is called once.
- **Contamination caveat.** The judge may recognise patches written by its own
  model family, or repositories and fixes it saw in training. The per-family
  breakdown below is the check: if the judge is much better on candidates from its
  own family, discount the result.

**To run it** (once, from a terminal). Either provider works; the notebook reads
whichever scores file exists.

```bash
# OpenAI judge (needs OPENAI_API_KEY); a GPT judge has the least overlap with the
# mostly non-GPT systems that wrote our candidates
.venv/bin/python pipeline/llm_judge.py run --provider openai --model gpt-5

# Anthropic judge (needs ANTHROPIC_API_KEY)
.venv/bin/python pipeline/llm_judge.py run --model claude-opus-5
```

`run` prints the token estimate, submits one batch, polls until it ends and
writes `data/llm_judge/<split>_scores.jsonl`. Use `estimate` first for the cost
table; for an OpenAI model pass `--price-in` and `--price-out` from the current
pricing page to see dollars.

**Laya, a System One decision model** (open weights, runs locally, no key), writes
the same format from its own environment:

```bash
./setup_and_run.sh --laya
.venv-laya/bin/python pipeline/laya_judge.py run --model multilingual --max-len 4096
```

The cells below evaluate every scores file found. Without any they print the
cost estimate and skip.

In [ ]:
# Experiment 2a: cost estimate and discovery of every judge scores file.
import glob
from llm_judge import JUDGE_DIR, estimate_cost, scope_rows

scope = scope_rows("val_test")
display(estimate_cost(scope))

# Any judge writes data/<split>/llm_judge/<split>_scores*.jsonl in the same format:
#   LLM judges (llm_judge.py)  -> <split>_scores.jsonl
#   Laya (laya_judge.py)       -> <split>_scores_laya-<checkpoint>.jsonl
judges = {}
for path in sorted(glob.glob(str(JUDGE_DIR / f"{TRUSTGATE_SPLIT}_scores*.jsonl"))):
    rows = [json.loads(l) for l in open(path) if l.strip()]
    if not rows:
        continue
    jd = pd.DataFrame(rows)
    name = str(jd["model"].dropna().iloc[0]) if jd["model"].notna().any() else os.path.basename(path)
    judges[name] = jd
    print(f"{name:<28} {len(jd):>5} scores, {jd['p_judge'].notna().sum():>5} usable  ({os.path.basename(path)})")
if not judges:
    print(f"\nNo judge scores under {JUDGE_DIR}. Run pipeline/run_experiment2.sh (LLM judge, needs a key) "
          "or .venv-laya/bin/python pipeline/laya_judge.py run (Laya, local, no key), then rerun from here.")

In [ ]:
# Experiment 2b: evaluate every judge alone, stacked with TrustGate, and per model family.
exp2_rows, exp2_family = [], {}
base = hgb().fit(train[BASE13], y_train)
p_tg_val_all = pd.Series(base.predict_proba(val[BASE13])[:, 1], index=val["candidate_id"].values)
p_tg_test_all = pd.Series(base.predict_proba(test[BASE13])[:, 1], index=test["candidate_id"].values)

def ev(name, yv, yt, pv, pt, groups, frame):
    lo, hi = grouped_bootstrap_auc(yt, pt, groups)
    a, r, met = choose_thresholds(yv, pv); g = gate_metrics(yt, pt, a, r)
    return {"experiment": name, "test_auc": roc_auc_score(yt, pt), "auc_lo": lo, "auc_hi": hi,
            "approve_share": g["approve_share"], "approve_precision": g["approve_precision"],
            "reject_share": g["reject_share"], "reject_precision": g["reject_precision"],
            "selection_resolve_rate": selection_rate(frame.assign(_s=pt), "_s")}

for name, jd in judges.items():
    js = jd.dropna(subset=["p_judge"]).drop_duplicates("candidate_id").set_index("candidate_id")["p_judge"]
    v = val[val["candidate_id"].isin(js.index)].copy(); t = test[test["candidate_id"].isin(js.index)].copy()
    if len(t) < 100 or t["label"].nunique() < 2:
        print(f"{name}: too few scored test rows ({len(t)}), skipped"); continue
    v["p_judge"] = v["candidate_id"].map(js).values; t["p_judge"] = t["candidate_id"].map(js).values
    v["p_tg"] = p_tg_val_all.loc[v["candidate_id"]].values; t["p_tg"] = p_tg_test_all.loc[t["candidate_id"]].values
    yv, yt = v["label"].astype(int).values, t["label"].astype(int).values
    stack = LogisticRegression().fit(v[["p_tg", "p_judge"]], yv)
    v["p_stack"] = stack.predict_proba(v[["p_tg", "p_judge"]])[:, 1]; t["p_stack"] = stack.predict_proba(t[["p_tg", "p_judge"]])[:, 1]
    g = t["instance_id"].values
    exp2_rows += [ev(f"TrustGate baseline (rows scored by {name})", yv, yt, v["p_tg"].values, t["p_tg"].values, g, t),
                  ev(f"{name} alone", yv, yt, v["p_judge"].values, t["p_judge"].values, g, t),
                  ev(f"TrustGate + {name}, logistic stack fit on val", yv, yt, v["p_stack"].values, t["p_stack"].values, g, t)]
    fam = t.assign(family=t["model_name"].str.extract(r"(claude|gpt|gemini|qwen|kimi|glm|llama)", flags=re.I)[0].str.lower().fillna("other"))
    exp2_family[name] = fam.groupby("family").apply(lambda d: pd.Series({
        "n": len(d),
        "judge_auc": roc_auc_score(d["label"], d["p_judge"]) if d["label"].nunique() == 2 else np.nan,
        "trustgate_auc": roc_auc_score(d["label"], d["p_tg"]) if d["label"].nunique() == 2 else np.nan}))

if exp2_rows:
    exp2 = pd.DataFrame(exp2_rows).set_index("experiment")
    display(exp2.round(3))
    for name, tab in exp2_family.items():
        print(f"\nContamination check for {name}: judge AUC by the model family that wrote the candidate")
        display(tab.round(3))
    print("\nExperiment 2 reading")
    for name in judges:
        key = f"TrustGate + {name}, logistic stack fit on val"
        if key not in exp2.index:
            continue
        s_ = exp2.loc[key, "test_auc"]; j_ = exp2.loc[f"{name} alone", "test_auc"]
        verdict = ("the shortfall was information; the patch-only approach was the constraint" if s_ >= 0.80
                   else "even a model reading the code does not predict the label well; approval needs execution or production labels")
        print(f"- {name}: alone {j_:.3f}, stacked {s_:.3f} -> {verdict}. "
              f"Approve precision {exp2.loc[key, 'approve_precision']:.3f} on {exp2.loc[key, 'approve_share']:.1%} (target 0.90); "
              f"selection {exp2.loc[key, 'selection_resolve_rate']:.3f} vs oracle {oracle:.3f}.")
else:
    exp2 = None
    print("Experiment 2 skipped: no judge scores yet.")

In [ ]:
# Save results for the write-up.
os.makedirs("docs", exist_ok=True)
out = {"split": TRUSTGATE_SPLIT, "reference": {"oracle": float(oracle), "best_single_system": float(best_system_rate), "best_system_name": best_system.index[0]},
       "experiment_1": json.loads(exp1.to_json(orient="index")),
       "experiment_2": None if exp2 is None else json.loads(exp2.to_json(orient="index"))}
path = f"docs/experiments_{TRUSTGATE_SPLIT}.json"
json.dump(out, open(path, "w"), indent=2)
print("Wrote", path)

## How to read the two results together

| Outcome | What it means | What to do next |
|---|---|---|
| Author prior adds several points; judge stack passes 0.80 | Missing information is partly reputation and partly semantics; both are recoverable | Pursue sharper agreement, code embeddings, and consider reputation where it is legitimate (production, where the tool is known) |
| Author prior adds several points; judge stack stays below 0.80 | Reputation matters, semantics do not help much | The remaining gap needs execution evidence, not smarter reading |
| Author prior adds little; judge stack passes 0.80 | Reputation is not the story; semantic understanding is | Code models are the priority |
| Author prior adds little; judge stack stays below 0.80 | The label is not predictable before the tests run | Keep TrustGate as a prioritiser; approval needs execution or production labels |

Whatever the outcome, record the numbers in `OBSERVATIONS.md` and the results deck. A null result here is as useful as a positive one.